# 📱 Google Drive Cloud Unzipper (Android Mobile Edition)
### Extract Gigabyte-sized ZIP / RAR / 7Z files inside Google Drive in seconds without downloading anything to your phone!

**Features:**
- ⚡ **Zero Mobile Data & Zero Phone Storage:** All extraction happens directly inside Google's high-speed cloud.
- 🏎️ **Ultra-Fast 7-Zip Engine:** Up to 10x faster than standard Python `zipfile`.
- 📂 **Auto-Detection:** Automatically scans your folders (like `GDFlix`) to find archives.
- 🔒 **Password Support:** Supports password-protected archives.
- 🗑️ **Optional Auto-Delete:** Automatically deletes the original `.zip` after successful extraction to save Drive storage.

In [ ]:
#@title ⚡ TAP HERE TO CONFIGURE & RUN EXTRACTION { display-mode: "form" }

import os
import sys
import glob
import shutil
import subprocess
from google.colab import drive

# =============================================================================
# ⚙️ USER SETTINGS (Tap to change)
# =============================================================================
#@markdown #### 📁 Source Settings
#@markdown Choose whether to specify an exact file or let the tool auto-find archives in a folder:
Mode = "Auto-Scan Folder (Newest Archive)" #@param ["Auto-Scan Folder (Newest Archive)", "Specify Exact Zip Path"]

#@markdown Folder to scan for archives (relative to MyDrive or full path):
Scan_Folder = "GDFlix" #@param {type:"string"}

#@markdown Exact ZIP path (only used if Mode is 'Specify Exact Zip Path'):
Exact_Archive_Path = "/content/drive/MyDrive/GDFlix/Elite.Force.S01.1080p.DS4K.SDR.10bit.NF.WEBRip.HIN-TAM-TEL-ENG-FRE.DDP5.1.x265.HEVC-OlaM.zip" #@param {type:"string"}

#@markdown #### 📂 Destination Settings
Destination_Folder = "/content/drive/MyDrive/MOVIES & WEB SERIES INFO" #@param {type:"string"}

#@markdown #### ⚙️ Extra Options
Delete_Archive_After_Extract = False #@param {type:"boolean"}
Archive_Password = "" #@param {type:"string"}
# =============================================================================

def ensure_drive_mounted():
    if not os.path.exists('/content/drive/MyDrive'):
        print("🔄 Mounting Google Drive...")
        drive.mount('/content/drive')
    else:
        print("✅ Google Drive already mounted.")

def resolve_path(path_str):
    path_str = path_str.strip().strip('"').strip("'")
    if not path_str.startswith('/content/drive/'):
        # Handle relative path like 'GDFlix' or 'MyDrive/GDFlix'
        if path_str.startswith('MyDrive/'):
            path_str = os.path.join('/content/drive', path_str)
        else:
            path_str = os.path.join('/content/drive/MyDrive', path_str)
    return os.path.abspath(path_str)

def find_target_archive():
    if Mode == "Specify Exact Zip Path":
        target = resolve_path(Exact_Archive_Path)
        if not os.path.exists(target):
            raise FileNotFoundError(f"❌ Specified file not found: {target}")
        return target
    else:
        folder = resolve_path(Scan_Folder)
        if not os.path.exists(folder):
            raise FileNotFoundError(f"❌ Scan folder does not exist: {folder}")
        
        # Search for all common archive extensions
        valid_exts = ('*.zip', '*.rar', '*.7z', '*.tar', '*.tar.gz', '*.tgz', '*.tar.bz2')
        found_files = []
        for ext in valid_exts:
            found_files.extend(glob.glob(os.path.join(folder, ext)))
            found_files.extend(glob.glob(os.path.join(folder, ext.upper())))
        
        if not found_files:
            raise FileNotFoundError(f"❌ No archives (.zip, .rar, .7z) found in {folder}")
        
        # Sort by last modified time (newest first)
        found_files.sort(key=os.path.getmtime, reverse=True)
        
        print(f"🔍 Found {len(found_files)} archive(s) in '{folder}'.")
        print(f"🎯 Selecting newest: {os.path.basename(found_files[0])}")
        return found_files[0]

def extract_fast(archive_path, dest_folder, password=""):
    # Make sure 7-Zip (p7zip-full) is installed for extreme speed and all format support
    subprocess.run(["apt-get", "install", "-y", "-qq", "p7zip-full"], check=False)
    
    os.makedirs(dest_folder, exist_ok=True)
    file_size_gb = os.path.getsize(archive_path) / (1024 ** 3)
    print(f"\n📦 File: {os.path.basename(archive_path)}")
    print(f"📏 Size: {file_size_gb:.2f} GB")
    print(f"📂 Destination: {dest_folder}")
    print("🚀 Extracting using high-speed cloud engine... Please wait a moment...")

    cmd = ["7z", "x", "-y", f"-o{dest_folder}"]
    if password:
        cmd.append(f"-p{password}")
    else:
        cmd.append("-p-")  # Do not prompt for password
    cmd.append(archive_path)

    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    
    for line in iter(process.stdout.readline, ''):
        clean_line = line.strip()
        if "%" in clean_line or "Extracting" in clean_line:
            sys.stdout.write(f"\r⏳ {clean_line[:80]}")
            sys.stdout.flush()
    
    process.wait()
    
    if process.returncode == 0:
        print("\n\n🎉 ✅ SUCCESS! All files extracted successfully!")
        if Delete_Archive_After_Extract:
            print(f"🗑️ Deleting archive to free space: {os.path.basename(archive_path)}")
            os.remove(archive_path)
            print("✅ Archive deleted successfully.")
    else:
        print(f"\n\n❌ Extraction failed with exit code {process.returncode}.")
        print("Check if the archive is corrupted or requires a password.")

# Execute Workflow
try:
    ensure_drive_mounted()
    archive = find_target_archive()
    dest = resolve_path(Destination_Folder)
    extract_fast(archive, dest, Archive_Password)
except Exception as e:
    print(f"\n⚠️ Error: {e}")


## 📱 Optional: Launch Full Touch-Friendly Mobile App Interface
Run the cell below if you want an interactive visual screen with a file browser, search bar, and 1-tap buttons on your phone.

In [ ]:
#@title 🌐 LAUNCH TOUCH-FRIENDLY WEB APP (Gradio UI)
!pip install -q gradio

import gradio as gr
import os, glob, subprocess
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

def scan_archives(folder_name):
    folder_name = folder_name.strip().strip('/')
    base_path = f"/content/drive/MyDrive/{folder_name}" if folder_name else "/content/drive/MyDrive"
    if not os.path.exists(base_path):
        return gr.Dropdown(choices=[], value=None), f"❌ Folder not found: {base_path}"
    
    exts = ('*.zip', '*.rar', '*.7z', '*.tar', '*.tgz')
    files = []
    for ext in exts:
        files.extend(glob.glob(os.path.join(base_path, ext)))
        files.extend(glob.glob(os.path.join(base_path, ext.upper())))
    
    files.sort(key=os.path.getmtime, reverse=True)
    file_names = [os.path.basename(f) for f in files]
    return gr.Dropdown(choices=file_names, value=file_names[0] if file_names else None), f"Found {len(files)} archives in {folder_name}"

def run_extract(folder_name, archive_name, dest_name, delete_zip, password):
    if not archive_name:
        return "❌ Please select an archive first!"
    
    base_src = f"/content/drive/MyDrive/{folder_name.strip('/')}" if folder_name else "/content/drive/MyDrive"
    archive_full = os.path.join(base_src, archive_name)
    dest_full = os.path.join("/content/drive/MyDrive", dest_name.strip('/'))
    
    os.makedirs(dest_full, exist_ok=True)
    cmd = ["7z", "x", "-y", f"-o{dest_full}"]
    if password:
        cmd.append(f"-p{password}")
    else:
        cmd.append("-p-")
    cmd.append(archive_full)
    
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode == 0:
        msg = f"✅ Successfully extracted to:\n{dest_full}"
        if delete_zip:
            os.remove(archive_full)
            msg += "\n🗑️ Original archive deleted to save Drive space."
        return msg
    else:
        return f"❌ Extraction failed:\n{res.stderr or res.stdout}"

with gr.Blocks(title="Drive Cloud Unzipper", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# ⚡ Google Drive Cloud Unzipper\nExtract large movies/series archives in the cloud without downloading anything!")
    
    with gr.Row():
        scan_input = gr.Textbox(value="GDFlix", label="Drive Source Folder (inside MyDrive)")
        scan_btn = gr.Button("🔍 Scan for Archives", variant="secondary")
    
    status_text = gr.Markdown("")
    archive_dropdown = gr.Dropdown(label="Select Archive File", choices=[], interactive=True)
    
    dest_input = gr.Textbox(value="MOVIES & WEB SERIES INFO", label="Destination Folder (inside MyDrive)")
    pass_input = gr.Textbox(value="", label="Password (optional)", type="password")
    delete_chk = gr.Checkbox(value=False, label="Delete archive after extraction to save Drive storage")
    
    extract_btn = gr.Button("🚀 EXTRACT NOW (Cloud-to-Cloud)", variant="primary")
    output_box = gr.Textbox(label="Output Log / Status", lines=4)
    
    scan_btn.click(scan_archives, inputs=[scan_input], outputs=[archive_dropdown, status_text])
    extract_btn.click(run_extract, inputs=[scan_input, archive_dropdown, dest_input, delete_chk, pass_input], outputs=[output_box])

demo.launch(share=True, inline=False)
